# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genius1oak/ConvexML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
!git clone https://github.com/genius1oak/ConvexML.git

Cloning into 'ConvexML'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 154 (delta 62), reused 89 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 1.95 MiB | 9.69 MiB/s, done.
Resolving deltas: 100% (62/62), done.


In [19]:
%cd ConvexML

/content/ConvexML/ConvexML


In [20]:
!ls data/raw

content_refresh_anonymized.csv


In [21]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [22]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print("\nDeclining rate:")
print(df["is_declining_label"].mean())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate:
0.5420666666666667


In [23]:
print(df[[
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "avg_position",
    "search_volume"
]].describe().T)

                          count         mean           std  min   25%    50%  \
days_since_last_update  30000.0    46.098300     42.078709  1.0  20.0   20.0   
impressions_90d         30000.0  5200.366300  16838.019547  1.0  81.0  731.0   
clicks_90d              30000.0    16.097333     75.076958  0.0   0.0    1.0   
pageviews_90d           30000.0    49.942467    152.101430  0.0   2.0    8.0   
sessions_90d            30000.0    37.066633    107.069131  1.0   2.0    7.0   
avg_position            30000.0    16.342380     15.216790  0.0   6.2   10.8   
search_volume           27532.0   158.882391   1518.270825  0.0   0.0   10.0   

                            75%       max  
days_since_last_update   104.00     373.0  
impressions_90d         3615.25  517715.0  
clicks_90d                 7.00    4178.0  
pageviews_90d             33.00    5998.0  
sessions_90d              27.00    4345.0  
avg_position              22.30     245.0  
search_volume             20.00   74000.0  


In [24]:
# Baseline candidate: stale + visible

df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)

df["baseline_test_score"] = (
    df["stale"] * df["visible"] * df["impressions_90d"]
)

# Rank highest score first
test_ranked = df.sort_values(
    "baseline_test_score",
    ascending=False
).reset_index(drop=True)

# Precision@20
top20 = test_ranked.head(20)
precision_at_20 = top20["is_declining_label"].mean()

print("Precision@20:", precision_at_20)
print("Base rate:", df["is_declining_label"].mean())

print("\nTop 20 declining labels:")
print(top20["is_declining_label"].value_counts())

Precision@20: 0.45
Base rate: 0.5420666666666667

Top 20 declining labels:
is_declining_label
0    11
1     9
Name: count, dtype: int64


In [25]:
import pandas as pd

# Rule A: stale content
df["score_stale"] = df["days_since_last_update"]

# Rule B: stale + impressions
df["score_stale_visible"] = (
    (df["days_since_last_update"] >= 90).astype(int)
    * df["impressions_90d"]
)

# Rule C: declining traffic pattern using only current/past 90-day metrics
df["traffic_drop"] = (
    (df["impressions_last_30d"] < df["impressions_prev_30d"]).astype(int)
)

df["score_traffic_drop"] = (
    df["traffic_drop"] * df["impressions_prev_30d"]
)

# Evaluate each rule at Top 20
rules = {
    "Stale only": "score_stale",
    "Stale + visible": "score_stale_visible",
    "Traffic drop": "score_traffic_drop"
}

base_rate = df["is_declining_label"].mean()

for name, score_col in rules.items():
    ranked = df.sort_values(score_col, ascending=False).head(20)
    precision = ranked["is_declining_label"].mean()

    print(f"{name}:")
    print(f"  Precision@20 = {precision:.3f}")
    print(f"  Base rate     = {base_rate:.3f}")
    print()

Stale only:
  Precision@20 = 0.500
  Base rate     = 0.542

Stale + visible:
  Precision@20 = 0.450
  Base rate     = 0.542

Traffic drop:
  Precision@20 = 0.650
  Base rate     = 0.542



In [26]:
# Rank using the traffic-drop baseline
ranked = df.sort_values(
    "score_traffic_drop",
    ascending=False
).reset_index(drop=True)

top20 = ranked.head(20).copy()

print(top20[[
    "content_id",
    "days_since_last_update",
    "impressions_last_30d",
    "impressions_prev_30d",
    "score_traffic_drop",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]].to_string(index=False))

          content_id  days_since_last_update  impressions_last_30d  impressions_prev_30d  score_traffic_drop trend_direction  trend_pct  is_declining_label
content_5fe46e04994d                     104                120791                218786              218786            down      -44.8                   1
content_aaef01a50def                      22                170559                177601              177601          stable       -4.0                   0
content_9532f197bbc8                     104                109317                174235              174235            down      -37.3                   1
content_2c2606c5d176                     104                104248                164079              164079            down      -36.5                   1
content_8c19996aa890                      20                 89463                161284              161284            down      -44.5                   1
content_1a9e894be2e2                      22                1079

### Baseline rule

Prioritize content where impressions in the most recent 30 days are lower than impressions in the previous 30 days. The score is the previous 30-day impression count, so content with a larger amount of previous visibility receives a higher review priority.

### Reason codes

* `TRAFFIC_DROP` — impressions decreased from the previous 30-day period.
* `NO_TRAFFIC_DROP` — impressions did not decrease, so the item receives no baseline priority.


In [27]:
# ML-07 baseline rule

df["traffic_drop"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

df["baseline_action_score"] = (
    df["traffic_drop"] * df["impressions_prev_30d"]
)

df["reason_code"] = "NO_TRAFFIC_DROP"
df.loc[df["traffic_drop"] == 1, "reason_code"] = "TRAFFIC_DROP"

print("Rule check:")
print("Traffic-drop items:", df["traffic_drop"].sum())
print("Non-drop items:", (df["traffic_drop"] == 0).sum())

Rule check:
Traffic-drop items: 19716
Non-drop items: 10284


### Ranked queue

The queue is ranked by the baseline action score from highest to lowest. A higher score means the content had more impressions in the previous 30-day period while also showing a drop in impressions in the latest 30 days.


In [28]:
# Rank all content and write the baseline queue

ranked = df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1

# Select the fields needed for the review queue
baseline_queue = ranked[[
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_code"
]].copy()

# Create output directory if needed
import os
os.makedirs("work/outputs", exist_ok=True)

# Write the ranked queue
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(baseline_queue):,}")

print("\nTop 20:")
print(baseline_queue.head(20).to_string(index=False))

Saved: work/outputs/baseline_action_score.csv
Rows: 30,000

Top 20:
 rank           content_id  baseline_action_score  reason_code
    1 content_5fe46e04994d                 218786 TRAFFIC_DROP
    2 content_aaef01a50def                 177601 TRAFFIC_DROP
    3 content_9532f197bbc8                 174235 TRAFFIC_DROP
    4 content_2c2606c5d176                 164079 TRAFFIC_DROP
    5 content_8c19996aa890                 161284 TRAFFIC_DROP
    6 content_1a9e894be2e2                 147889 TRAFFIC_DROP
    7 content_4c36c775b818                 125416 TRAFFIC_DROP
    8 content_cb112fce36be                 124500 TRAFFIC_DROP
    9 content_44e481c8f55b                 118137 TRAFFIC_DROP
   10 content_c8e9d6ab9013                 111885 TRAFFIC_DROP
   11 content_b28d1efd668f                 110679 TRAFFIC_DROP
   12 content_cea79ef51519                 104722 TRAFFIC_DROP
   13 content_89e84d699e9e                  97854 TRAFFIC_DROP
   14 content_73c54f78c06a                  97200 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [29]:
# ML-07 — Top-20 review

top20_review = ranked.head(20).copy()

# Action is the same baseline action for every traffic-drop pick.
top20_review["action"] = "REVIEW_CONTENT"

# Confidence is based on the size of the observed impression drop.
top20_review["drop_pct"] = (
    (top20_review["impressions_prev_30d"] -
     top20_review["impressions_last_30d"])
    / top20_review["impressions_prev_30d"]
) * 100

def confidence_note(drop_pct):
    if drop_pct >= 30:
        return "Higher confidence: observed impression drop is >=30%."
    elif drop_pct >= 15:
        return "Moderate confidence: observed impression drop is 15–29%."
    else:
        return "Lower confidence: observed impression drop is <15%."

top20_review["confidence_note"] = (
    top20_review["drop_pct"].apply(confidence_note)
)

top20_review["what_would_make_it_wrong"] = (
    "The impression drop could be temporary, seasonal, "
    "tracking-related, or caused by another factor not captured here."
)

# Display the required review
print(top20_review[[
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]].to_string(index=False))

 rank           content_id         action  reason_code                                          confidence_note                                                                                           what_would_make_it_wrong
    1 content_5fe46e04994d REVIEW_CONTENT TRAFFIC_DROP    Higher confidence: observed impression drop is >=30%. The impression drop could be temporary, seasonal, tracking-related, or caused by another factor not captured here.
    2 content_aaef01a50def REVIEW_CONTENT TRAFFIC_DROP      Lower confidence: observed impression drop is <15%. The impression drop could be temporary, seasonal, tracking-related, or caused by another factor not captured here.
    3 content_9532f197bbc8 REVIEW_CONTENT TRAFFIC_DROP    Higher confidence: observed impression drop is >=30%. The impression drop could be temporary, seasonal, tracking-related, or caused by another factor not captured here.
    4 content_2c2606c5d176 REVIEW_CONTENT TRAFFIC_DROP    Higher confidence: observed impres

In [30]:
# ML-07 — Baseline evaluation

top20 = ranked.head(20)

precision_at_20 = top20["is_declining_label"].mean()
base_rate = df["is_declining_label"].mean()
lift = precision_at_20 - base_rate

print(f"Precision@20: {precision_at_20:.3f}")
print(f"Base rate:    {base_rate:.3f}")
print(f"Lift:         {lift:.3f}")

Precision@20: 0.650
Base rate:    0.542
Lift:         0.108


### Weak picks

Some top-ranked items can still be false positives. In the Top-20 review, the baseline selected several items that are not marked as declining, showing that an observed traffic drop does not always correspond to the decline label. These are useful weak picks to review rather than treating the baseline as a prediction.

### Leakage check

The baseline score uses only `impressions_last_30d` and `impressions_prev_30d`. It does not use `trend_direction`, `trend_pct`, or `is_declining_label` as scoring inputs. No product flags, client names, private queries, or future-window variables are used in the rule.


In [31]:
# ML-07 — Weak picks and leakage checks

# Weak picks: Top-20 items that are not marked as declining
weak_picks = top20_review[
    top20_review["is_declining_label"] == 0
].copy()

print("Weak picks in Top-20:", len(weak_picks))

print("\nWeak picks:")
print(weak_picks[[
    "rank",
    "content_id",
    "impressions_last_30d",
    "impressions_prev_30d",
    "drop_pct",
    "trend_direction",
    "is_declining_label"
]].to_string(index=False))


# Leakage check
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

scoring_features = [
    "impressions_last_30d",
    "impressions_prev_30d"
]

print("\nLeakage check:")
print("Scoring features:", scoring_features)
print("Forbidden label/future-derived fields:", forbidden_features)

assert "trend_direction" not in scoring_features
assert "trend_pct" not in scoring_features
assert "is_declining_label" not in scoring_features

print("PASS: forbidden label-derived fields are not used in the baseline score.")

Weak picks in Top-20: 7

Weak picks:
 rank           content_id  impressions_last_30d  impressions_prev_30d  drop_pct trend_direction  is_declining_label
    2 content_aaef01a50def                170559                177601  3.965068          stable                   0
    9 content_44e481c8f55b                104458                118137 11.578930          stable                   0
   11 content_b28d1efd668f                 91655                110679 17.188446          stable                   0
   13 content_89e84d699e9e                 83350                 97854 14.822082          stable                   0
   14 content_73c54f78c06a                 85885                 97200 11.640947          stable                   0
   16 content_aa4baf490b43                 82500                 95175 13.317573          stable                   0
   18 content_8e7ba84a972b                 75144                 91776 18.122385          stable                   0

Leakage check:
Scoring fea

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.